# Autoencoder: benign training and injection evaluation

This walkthrough trains host-specific autoencoders on the longitudinal benign files, then scores the controlled clean and injection-labelled snapshots.

The frozen `ae_score` and `ae_flag` columns remain the published result; retraining can differ. Download the five CSV files from Mendeley Data: https://doi.org/10.17632/gj8vp49xm5.1

## Step 1. Imports

In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Step 2. Find the GitHub repo root

In [ ]:
REPO_CANDIDATES = [Path("..").resolve(), Path(".").resolve()]
REPO_ROOT = next(path for path in REPO_CANDIDATES if (path / "metadata" / "feature_schema.json").is_file())
sys.path.insert(0, str(REPO_ROOT / "scripts"))
print(REPO_ROOT)

## Step 3. Point at the extracted Mendeley Data folder

By default, place the extracted files at `data/wasp/data/` in the repository. Alternatively, set `WASP_DATA_ROOT` to the extracted `data/` directory before starting Jupyter.

In [ ]:
DATA_ROOT = Path(os.environ.get("WASP_DATA_ROOT", REPO_ROOT / "data" / "wasp" / "data")).expanduser().resolve()
if not (DATA_ROOT / "controlled_paired_dataset").is_dir() or not (DATA_ROOT / "longitudinal_benign_dataset").is_dir():
    raise FileNotFoundError("Dataset not found. Set WASP_DATA_ROOT to the extracted data/ directory.")
EXTRACT_ROOT = DATA_ROOT.parent
OUTPUT_DIR = REPO_ROOT / "outputs" / "notebooks"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BLUE = "#2563EB"
PURPLE = "#7C3AED"
RED = "#E11D48"
CLASS_COLORS = {"benign": BLUE, "injected": RED}
MODEL_COLORS = {"retrained": BLUE, "frozen": PURPLE}


def style_figure(fig, title, width=860, height=540, **extra):
    fig.update_layout(
        title=dict(text=title, x=0.02, xanchor="left", font=dict(size=18, color="#0F172A")),
        font=dict(family="Source Sans 3, Helvetica Neue, Arial, sans-serif", size=14, color="#1E293B"),
        paper_bgcolor="#F8FAFC",
        plot_bgcolor="#FFFFFF",
        width=width,
        height=height,
        legend=dict(bgcolor="rgba(255,255,255,0.92)", bordercolor="rgba(15,23,42,0.08)", borderwidth=1),
        margin=dict(l=64, r=28, t=64, b=56),
        hoverlabel=dict(bgcolor="#0F172A", font=dict(size=12, color="#F8FAFC")),
        **extra,
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(148,163,184,0.28)", zeroline=False, linecolor="#CBD5E1")
    fig.update_yaxes(showgrid=True, gridcolor="rgba(148,163,184,0.28)", zeroline=False, linecolor="#CBD5E1")
    return fig


def show_plotly(fig, stem):
    fig.write_html(OUTPUT_DIR / f"{stem}.html", include_plotlyjs="cdn")
    try:
        fig.write_image(OUTPUT_DIR / f"{stem}.png", scale=2)
    except Exception as exc:
        print(f"PNG export skipped for {stem}.png ({type(exc).__name__})")
    fig.show()

print(DATA_ROOT)
print(EXTRACT_ROOT)

## Step 4. Choose hosts and a verification epoch budget

Set `EPOCHS_OVERRIDE = None` to use the published 100 epochs.

In [ ]:
HOSTS = ["windows10", "win11_x64", "win2016_dc"]
EPOCHS_OVERRIDE = None
print(HOSTS)
print("epochs override:", EPOCHS_OVERRIDE)

## Step 5. Load the Windows 10 benign file

In [ ]:
benign = {}
benign["windows10"] = pd.read_csv(DATA_ROOT / "longitudinal_benign_dataset" / "windows10.csv", low_memory=False)
print(benign["windows10"].shape)
benign["windows10"][["binary_id", "run_group_id", "working_set_bytes"]].head(3)

## Step 6. Load the Windows 11 benign file

In [ ]:
benign["win11_x64"] = pd.read_csv(DATA_ROOT / "longitudinal_benign_dataset" / "win11_x64.csv", low_memory=False)
print(benign["win11_x64"].shape)

## Step 7. Load the Server 2016 benign file

In [ ]:
benign["win2016_dc"] = pd.read_csv(DATA_ROOT / "longitudinal_benign_dataset" / "win2016_dc.csv", low_memory=False)
print(benign["win2016_dc"].shape)
print("total benign snapshots:", sum(len(frame) for frame in benign.values()))

## Step 8. Summarise the three benign files

In [ ]:
pd.DataFrame(
    [
        {
            "host": host,
            "rows": len(frame),
            "runs": frame["run_group_id"].nunique(),
            "binaries": frame["binary_id"].nunique(),
        }
        for host, frame in benign.items()
    ]
)

## Step 9. Load the clean-condition test file

These rows are evaluation only. They are not used for training.

In [ ]:
clean = pd.read_csv(DATA_ROOT / "controlled_paired_dataset" / "controlled_clean_snapshots.csv", low_memory=False)
print(clean.shape)

## Step 10. Load the injection-labelled test file

In [ ]:
injected = pd.read_csv(DATA_ROOT / "controlled_paired_dataset" / "injection_labelled_snapshots.csv", low_memory=False)
print(injected.shape)

## Step 11. Join the evaluation conditions

In [ ]:
controlled = pd.concat([clean, injected], ignore_index=True)
print("evaluation rows:", len(controlled))
print("pairs:", controlled["pair_group_id"].nunique())
print("scoreable pairs:", controlled.loc[controlled["score_status"].eq("scoreable"), "pair_group_id"].nunique())

## Step 12. Load the published training config

In [ ]:
config = json.loads((REPO_ROOT / "config" / "model.json").read_text(encoding="utf-8"))
if EPOCHS_OVERRIDE is not None:
    config["training"]["epochs"] = int(EPOCHS_OVERRIDE)
print(config["architecture"])
print(config["training"])

## Step 13. Load host model receipts

In [ ]:
receipts = json.loads((REPO_ROOT / "metadata" / "model_bundle_receipts.json").read_text(encoding="utf-8"))
receipt_map = {item["host_key"]: item for item in receipts["bundles"]}
feature_names = list(receipt_map["windows10"]["feature_names"])
print(len(feature_names), "inputs")
print(feature_names[:8])

## Step 14. Select the training device

The trainer tries CUDA, then Apple MPS, then CPU. It keeps going if a backend is missing or unusable. Set `WASP_DEVICE=cpu` to force CPU.

In [ ]:
from model import select_device
from train_autoencoder import train_host

device = select_device("auto")
ae_output = OUTPUT_DIR / "autoencoder_bundles"
ae_output.mkdir(parents=True, exist_ok=True)
print("device:", device)
print(ae_output)

## Step 15. Train the Windows 10 autoencoder

Training uses only `longitudinal_benign_dataset/windows10.csv`.

In [ ]:
summary_windows10 = train_host(EXTRACT_ROOT, ae_output, "windows10", config, receipt_map["windows10"], device, None)
print(summary_windows10["train_rows"], "train rows")
print(summary_windows10["validation_rows"], "validation rows")
print(summary_windows10["epochs_ran"], "epochs")
print("val mse", round(summary_windows10["best_validation_mse"], 6))

## Step 16. Train the Windows 11 autoencoder

In [ ]:
summary_win11 = train_host(EXTRACT_ROOT, ae_output, "win11_x64", config, receipt_map["win11_x64"], device, None)
print(summary_win11["train_rows"], "train rows")
print(summary_win11["validation_rows"], "validation rows")
print(summary_win11["epochs_ran"], "epochs")
print("val mse", round(summary_win11["best_validation_mse"], 6))

## Step 17. Train the Server 2016 autoencoder

In [ ]:
summary_win2016 = train_host(EXTRACT_ROOT, ae_output, "win2016_dc", config, receipt_map["win2016_dc"], device, None)
print(summary_win2016["train_rows"], "train rows")
print(summary_win2016["validation_rows"], "validation rows")
print(summary_win2016["epochs_ran"], "epochs")
print("val mse", round(summary_win2016["best_validation_mse"], 6))

## Step 18. Summarise the three trained bundles

In [ ]:
summaries = [summary_windows10, summary_win11, summary_win2016]
pd.DataFrame(
    [
        {
            "host": item["host"],
            "train_rows": item["train_rows"],
            "validation_rows": item["validation_rows"],
            "epochs_ran": item["epochs_ran"],
            "best_validation_mse": item["best_validation_mse"],
            "binaries": len(item["binary_vocabulary"]),
            "skipped": len(item["skipped_binaries"]),
        }
        for item in summaries
    ]
)

## Step 19. Plot benign training curves

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=HOSTS, shared_yaxes=True, horizontal_spacing=0.06)
for col, host in enumerate(HOSTS, start=1):
    hist = pd.DataFrame(json.loads((ae_output / host / "history.json").read_text(encoding="utf-8")))
    fig.add_trace(go.Scatter(x=hist["epoch"], y=hist["training_mse"], name="train", legendgroup="train", showlegend=col == 1, line=dict(color=BLUE, width=2.8, shape="spline")), row=1, col=col)
    fig.add_trace(go.Scatter(x=hist["epoch"], y=hist["validation_mse"], name="validation", legendgroup="validation", showlegend=col == 1, line=dict(color=PURPLE, width=2.8, shape="spline")), row=1, col=col)
    fig.update_xaxes(title="epoch", row=1, col=col)
style_figure(fig, "Benign training curves", width=1100, height=420)
fig.update_yaxes(title="MSE", row=1, col=1)
show_plotly(fig, "ae_training_history")

## Step 20. Load helpers used to score the evaluation files

In [ ]:
from train_autoencoder import _snapshot_mse, _transform
from model import AutoencoderArchitecture, BinaryConditionedAutoencoder
import torch

## Step 21. Load one trained bundle

In [ ]:
def load_bundle(host: str):
    meta = json.loads((ae_output / host / "metadata.json").read_text(encoding="utf-8"))
    prep = np.load(ae_output / host / "preprocessor.npz", allow_pickle=True)
    payload = torch.load(ae_output / host / "model.pt", map_location=device)
    model = BinaryConditionedAutoencoder(AutoencoderArchitecture(**payload["architecture"])).to(device)
    model.load_state_dict(payload["state_dict"])
    model.eval()
    return meta, prep, model

meta, prep, model = load_bundle("windows10")
print("windows10 vocabulary:", len(meta["binary_vocabulary"]))

## Step 22. Score one host of evaluation rows

In [ ]:
def score_host(host_frame: pd.DataFrame, host: str) -> pd.DataFrame:
    meta, prep, model = load_bundle(host)
    names = list(meta["feature_names"])
    transforms = dict(meta["feature_transforms"])
    vocab = {binary_id: index for index, binary_id in enumerate(meta["binary_vocabulary"])}
    raw = host_frame[names].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    transformed = np.column_stack(
        [_transform(raw[:, index], transforms[name], float(config["logit_epsilon"])) for index, name in enumerate(names)]
    )
    missing_rows, missing_cols = np.where(~np.isfinite(transformed))
    transformed[missing_rows, missing_cols] = prep["medians"][missing_cols]
    features = np.clip((transformed - prep["centre"]) / prep["scale"], -float(config["scaled_clip"]), float(config["scaled_clip"])).astype(np.float32)
    binary_ids = host_frame["binary_id"].astype(str)
    enrolled = binary_ids.isin(vocab)
    scores = np.full(len(host_frame), np.nan, dtype=np.float64)
    if enrolled.any():
        indices = binary_ids[enrolled].map(vocab).to_numpy(dtype=np.int64)
        scores[enrolled.to_numpy()] = _snapshot_mse(model, features[enrolled.to_numpy()], indices, device)
    part = host_frame.copy()
    part["retrain_score"] = scores
    part["retrain_threshold"] = [meta["thresholds"].get(str(binary_id), np.nan) for binary_id in binary_ids]
    part["retrain_enrolled"] = enrolled.to_numpy()
    part["retrain_flag"] = part["retrain_score"].ge(part["retrain_threshold"])
    return part

print("helper ready")

## Step 23. Score Windows 10 evaluation rows

In [ ]:
scored_parts = [score_host(controlled.loc[controlled["host_model"].eq("windows10")], "windows10")]
print(len(scored_parts[0]), "windows10 evaluation rows")

## Step 24. Score Windows 11 evaluation rows

In [ ]:
scored_parts.append(score_host(controlled.loc[controlled["host_model"].eq("win11_x64")], "win11_x64"))
print(len(scored_parts[-1]), "win11_x64 evaluation rows")

## Step 25. Score Server 2016 evaluation rows

In [ ]:
scored_parts.append(score_host(controlled.loc[controlled["host_model"].eq("win2016_dc")], "win2016_dc"))
scored = pd.concat(scored_parts, ignore_index=True)
print("all evaluation rows:", len(scored))
print("enrolled rows:", int(scored["retrain_enrolled"].sum()))

## Step 26. Inspect a few scored rows

In [ ]:
scored.loc[
    scored["retrain_enrolled"],
    ["host_model", "binary_id", "class_label", "retrain_score", "retrain_threshold", "retrain_flag", "ae_flag"],
].head(8)

## Step 27. Snapshot confusion for the retrained model

In [ ]:
enrolled = scored.loc[scored["retrain_enrolled"]].copy()
enrolled["frozen_flag"] = pd.to_numeric(enrolled["ae_flag"], errors="coerce")
pd.crosstab(enrolled["class_label"], enrolled["retrain_flag"], dropna=False)

## Step 28. Snapshot confusion for the frozen published scores

In [ ]:
pd.crosstab(enrolled["class_label"], enrolled["frozen_flag"], dropna=False)

## Step 29. Agreement with frozen decisions

In [ ]:
agree = enrolled["retrain_flag"].astype(int).eq(enrolled["frozen_flag"].fillna(-1).astype(int))
print(f"{int(agree.sum())}/{len(enrolled)}")

## Step 30. Pair-level complete detection

In [ ]:
pair_rows = []
for pair_id, pair in enrolled.groupby("pair_group_id"):
    injected_pair = pair.loc[pair["label_injected"].eq(1)]
    clean_pair = pair.loc[pair["label_injected"].eq(0)]
    pair_rows.append(
        {
            "pair_group_id": pair_id,
            "payload_family": pair["payload_family"].iloc[0],
            "technique_label": pair["technique_label"].iloc[0],
            "retrain_complete": bool(injected_pair["retrain_flag"].all()) if len(injected_pair) else False,
            "retrain_clean_alert": bool(clean_pair["retrain_flag"].any()) if len(clean_pair) else False,
            "frozen_complete": bool(injected_pair["frozen_flag"].eq(1).all()) if len(injected_pair) else False,
            "frozen_clean_alert": bool(clean_pair["frozen_flag"].eq(1).any()) if len(clean_pair) else False,
        }
    )
pairs = pd.DataFrame(pair_rows)
print("retrained complete:", f"{int(pairs['retrain_complete'].sum())}/{len(pairs)}")
print("frozen complete:", f"{int(pairs['frozen_complete'].sum())}/{len(pairs)}")
print("published reference: 59/60 complete, 7/1736 clean snapshot alerts")
pairs

## Step 31. Compare score distributions

In [ ]:
panels = (("retrain_score", "Retrained reconstruction error"), ("ae_score", "Frozen published score"))
fig = make_subplots(rows=1, cols=2, subplot_titles=[title for _, title in panels], shared_yaxes=True, horizontal_spacing=0.08)
for col, (name, _title) in enumerate(panels, start=1):
    for label in ("benign", "injected"):
        values = pd.to_numeric(enrolled.loc[enrolled["class_label"].eq(label), name], errors="coerce").dropna()
        fig.add_trace(
            go.Histogram(
                x=np.log10(np.clip(values, 1e-12, None)),
                name=label,
                legendgroup=label,
                showlegend=col == 1,
                marker=dict(color=CLASS_COLORS[label], line=dict(width=0)),
                opacity=0.72,
                nbinsx=40,
            ),
            row=1,
            col=col,
        )
        fig.update_xaxes(title="log10(score)", row=1, col=col)
style_figure(fig, "Score distributions", width=980, height=480, barmode="overlay")
fig.update_yaxes(title="Snapshots", row=1, col=1)
show_plotly(fig, "ae_score_histograms")

## Step 32. Confusion-matrix figures

Same enrolled evaluation rows as the tables above: retrained flags on the left, frozen published flags on the right.

In [ ]:
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve

y_true = enrolled["label_injected"].astype(int).to_numpy()
y_retrain = enrolled["retrain_flag"].astype(int).to_numpy()
frozen_ok = enrolled["frozen_flag"].notna()
y_frozen = enrolled.loc[frozen_ok, "frozen_flag"].astype(int).to_numpy()
y_true_frozen = enrolled.loc[frozen_ok, "label_injected"].astype(int).to_numpy()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Retrained autoencoder", "Frozen published scores"], horizontal_spacing=0.12)
for col, (truth, pred) in enumerate(((y_true, y_retrain), (y_true_frozen, y_frozen)), start=1):
    matrix = confusion_matrix(truth, pred, labels=[0, 1])
    fig.add_trace(
        go.Heatmap(
            z=matrix,
            x=["predicted clean", "predicted injected"],
            y=["true clean", "true injected"],
            colorscale=[[0, "#EEF2FF"], [0.5, "#A78BFA"], [1, "#5B21B6"]],
            showscale=col == 2,
            text=matrix,
            texttemplate="%{text}",
            textfont=dict(size=16, color="#0F172A"),
            hovertemplate="%{y} / %{x}: %{z}<extra></extra>",
        ),
        row=1,
        col=col,
    )
style_figure(fig, "Snapshot confusion matrices", width=980, height=460)
fig.update_yaxes(autorange="reversed")
show_plotly(fig, "ae_confusion_matrices")

## Step 33. Snapshot ROC curves

Higher reconstruction error is treated as the positive (injected) score. Both curves use enrolled evaluation rows only.

In [ ]:
fig = go.Figure()
for score_col, title, colour in (("retrain_score", "Retrained", BLUE), ("ae_score", "Frozen published", PURPLE)):
    scores = pd.to_numeric(enrolled[score_col], errors="coerce")
    mask = np.isfinite(scores.to_numpy())
    fpr, tpr, _ = roc_curve(y_true[mask], scores.to_numpy()[mask])
    fig.add_trace(
        go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{title}  AUC={roc_auc_score(y_true[mask], scores.to_numpy()[mask]):.3f}",
            line=dict(color=colour, width=3.2, shape="spline"),
            hovertemplate="FPR=%{x:.3f}<br>TPR=%{y:.3f}<extra>%{fullData.name}</extra>",
        )
    )
style_figure(fig, "Snapshot ROC")
fig.update_xaxes(title="False positive rate", range=[-0.02, 1.02])
fig.update_yaxes(title="True positive rate", range=[-0.02, 1.04])
show_plotly(fig, "ae_roc_curves")

## Step 34. Snapshot precision-recall curves

In [ ]:
fig = go.Figure()
for score_col, title, colour in (("retrain_score", "Retrained", BLUE), ("ae_score", "Frozen published", PURPLE)):
    scores = pd.to_numeric(enrolled[score_col], errors="coerce")
    mask = np.isfinite(scores.to_numpy())
    precision, recall, _ = precision_recall_curve(y_true[mask], scores.to_numpy()[mask])
    ap = average_precision_score(y_true[mask], scores.to_numpy()[mask])
    fig.add_trace(
        go.Scatter(
            x=recall,
            y=precision,
            mode="lines",
            name=f"{title}  AP={ap:.3f}",
            line=dict(color=colour, width=3.2, shape="spline"),
            hovertemplate="Recall=%{x:.3f}<br>Precision=%{y:.3f}<extra>%{fullData.name}</extra>",
        )
    )
style_figure(fig, "Snapshot precision-recall")
fig.update_xaxes(title="Recall", range=[-0.02, 1.02])
fig.update_yaxes(title="Precision", range=[0, 1.05])
show_plotly(fig, "ae_pr_curves")

## Step 35. Complete pair detection by technique

A pair is complete only when every injection-labelled snapshot exceeds its threshold.

In [ ]:
summary = pairs.groupby("technique_label")[["retrain_complete", "frozen_complete"]].mean().sort_index().reset_index()
fig = go.Figure()
fig.add_trace(go.Bar(x=summary["technique_label"], y=summary["retrain_complete"], name="retrained", marker=dict(color=BLUE, line=dict(width=0))))
fig.add_trace(go.Bar(x=summary["technique_label"], y=summary["frozen_complete"], name="frozen published", marker=dict(color=PURPLE, line=dict(width=0))))
style_figure(fig, "Pair-level complete detection", width=920, height=500, barmode="group")
fig.update_yaxes(title="Complete injected-pair rate", range=[0, 1.08])
show_plotly(fig, "ae_pair_detection_by_technique")
summary

## Step 36. Score versus per-binary threshold

Points above the dashed line are flagged. Clean and injected evaluation snapshots are overlaid.

In [ ]:
panels = (("retrain_score", "retrain_threshold", "Retrained"), ("ae_score", "ae_threshold", "Frozen published"))
fig = make_subplots(rows=1, cols=2, subplot_titles=[title for _, _, title in panels], shared_xaxes=True, shared_yaxes=True, horizontal_spacing=0.08)
for col, (score_col, threshold_col, _title) in enumerate(panels, start=1):
    for label in ("benign", "injected"):
        part = enrolled.loc[enrolled["class_label"].eq(label)]
        x = pd.to_numeric(part[threshold_col], errors="coerce")
        y = pd.to_numeric(part[score_col], errors="coerce")
        keep = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
        fig.add_trace(
            go.Scatter(
                x=np.log10(x[keep]),
                y=np.log10(y[keep]),
                mode="markers",
                name=label,
                legendgroup=label,
                showlegend=col == 1,
                marker=dict(size=7, color=CLASS_COLORS[label], opacity=0.45, line=dict(width=0)),
            ),
            row=1,
            col=col,
        )
    fig.update_xaxes(title="log10(threshold)", row=1, col=col)
    fig.update_yaxes(title="log10(score)" if col == 1 else None, row=1, col=col)
style_figure(fig, "Score versus per-binary threshold", width=1000, height=500)
show_plotly(fig, "ae_score_vs_threshold")

## Step 37. Snapshot rates by host

True-positive rate on injected rows and false-positive rate on clean rows, for each host model.

In [ ]:
rate_rows = []
for host, part in enrolled.groupby("host_model"):
    injected = part.loc[part["label_injected"].eq(1)]
    clean = part.loc[part["label_injected"].eq(0)]
    frozen_injected = injected.loc[injected["frozen_flag"].notna()]
    frozen_clean = clean.loc[clean["frozen_flag"].notna()]
    rate_rows.append(
        {
            "host": host,
            "retrain_tpr": float(injected["retrain_flag"].mean()) if len(injected) else np.nan,
            "retrain_fpr": float(clean["retrain_flag"].mean()) if len(clean) else np.nan,
            "frozen_tpr": float(frozen_injected["frozen_flag"].eq(1).mean()) if len(frozen_injected) else np.nan,
            "frozen_fpr": float(frozen_clean["frozen_flag"].eq(1).mean()) if len(frozen_clean) else np.nan,
        }
    )
rates = pd.DataFrame(rate_rows)
fig = go.Figure()
for column, label, colour in (
    ("retrain_tpr", "retrained TPR", BLUE),
    ("frozen_tpr", "frozen TPR", PURPLE),
    ("retrain_fpr", "retrained FPR", RED),
    ("frozen_fpr", "frozen FPR", "#FB7185"),
):
    fig.add_trace(go.Bar(x=rates["host"], y=rates[column], name=label, marker=dict(color=colour, line=dict(width=0))))
style_figure(fig, "Snapshot detection rates by host", width=920, height=500, barmode="group")
fig.update_yaxes(title="Rate", range=[0, 1.08])
show_plotly(fig, "ae_host_detection_rates")
rates